# TROPOMI HDO search

Scratch notebook for browsing the SRON TROPOMI server (https://tropomi.grid.surfsara.nl/) with the same WebDAV listing `tropomi_download.py` uses, and for looking inside a file. Nothing here downloads.

In [ ]:
import os
import requests
import xarray as xr

from utils.path_utils import load_config
from tropomi_download import propfind, list_months, list_month_files, parse_date_range

settings = load_config('config/config_download.yaml')
session = requests.Session()

## Product layout: `<product>/YYYY/MM/DD/<orbit>/s5p_l2_<product>_<version>_<orbit>.nc`

In [ ]:
# years and months on the server, optionally limited to a date range
months = list_months(session, settings, parse_date_range(['2025-01-01', '2025-12-31']))
months

In [ ]:
# one orbit directory, every property the server reports
url = 'https://tropomi.grid.surfsara.nl/hdo/2024/05/01/33931/'
propfind(session, url, '1', retries=3)

In [ ]:
# every file in one month, with sizes; versions and others tally what is not selected
from collections import Counter
versions, others = Counter(), []
files = list_month_files(session, settings, 2025, 11, None, versions, others)
print(len(files), 'files', sum(f[2] for f in files) / 1024**4, 'TiB', dict(versions), len(others), 'others')
files[:3]

## Inside a file

No root variables; everything is in groups: `target_product` (`deltad`, `hdo_column`, `h2o_column`, ...), `diagnostics` (`quality_flag`, ...), `instrument` (`latitude_center`, `longitude_center`, `time`, ...), `meteo` and `side_product`. Dimension `nobs` is the sounding.

In [ ]:
path = os.path.join(
    settings['directories']['download'],
    'tropomi.grid.surfsara.nl/hdo/2024/05/01/33931/s5p_l2_hdo_0003_33931.nc',
)
target = xr.open_dataset(path, group='target_product')
instrument = xr.open_dataset(path, group='instrument')
target